# Exploration strategies for DQN algorithm

In this assignment we are interested in exploration strategies that can be combined with Q-learning.
Q-learning is an off-policy algorithm, which means that the data for the algorithm can be collected by a different policy (called behavioural policy) that the one the algorithm learns.

Here we come across a classical trade-off in reinforcement learning, called exploration-exploitation trade-off. On the one hand, our behavioural policy should try out new state-action pairs to gain knowledge about their returns. On the other hand, when our estimate of returns is good enough, we would like to follow the state-action pairs with the highest estimated returns.

We will be operating on DQN [(Mnih 2014)](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf) algorithm and analyzing epsilon-greedy strategy, boltzmann and max-boltzmann strategy and combination of epsilon-greedy and boltzmann.
We evaluate performance of DQN variants on the Lunar Lander environment.

We provide an implementation of the DQN algorithm with random exploration strategy.
Your goal is to implement the exploration variants by overriding appropriate methods of the provided class.


## Grading

To obtain the points for the assignment You need to provide the implementation of exploration techniques AND report with plots and conclusions.
Measuring sensitivity means that You should at least examine one reasonably lower and one reasonably greater value of the considered hyperparameter (or the pair of hyperparameters).


1. Implement epsilon-greedy strategy and investigate hyperparameter sensitivity (1 point).
2. Implement epsilon-greedy strategy with epsilon annealing and investigate hyperparameter sensitivity (1 point).
3. Implement boltzmann strategy and investigate hyperparameter sensitivity (1 point).
4. Implement boltzmann strategy with temperature annealing and investigate hyperparameter sensitivity (1 point).
5. Implement max-boltzmann strategy and investigate hyperparameter sensitivity (1 point).
6. Implement max-boltzmann strategy with temperature annealing and investigate hyperparameter sensitivity (1 point).
7. Implement combination of epsilon-greedy with epsilon annealing and boltzmann strategy and investigate hyperparameter sensitivity (1 point)
8. (*) Bonus: propose another reasonable approach to combine epsilon-greedy with epsilon annealing strategy and boltzmann strategy and/or another reasonable strategy of temperature annealing for the boltzmann strategy (2 points).
9. Compare methods, present plots and conclusions in a clear manner (3 points).

You can obtain max 10 points, bonus points increase Your score, if You lose points in some other tasks.

Here we import necessary libraries.

In [1]:
# !apt-get install swig
# !pip install gymnasium[box2d]

In [2]:
import torch
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

Here we set hyperparameters of the training, set seeds for reproducibility and set weights initialization.
Although for debugging it might be useful to operate on a smaller number of training_steps, seeds etc., in the final evaluation DO NOT CHANGE these parameters.

In [ ]:
class parse_args:
	def __init__(self):
		self.n_seeds = 6
		self.n_evaluate_episodes = 5
		self.n_training_steps = 100000
		self.buffer_size = 10000
		self.init_steps = 10000
		self.target_update_freq = 50
		self.eval_freq = 1000
		self.gym_id = "LunarLander-v3"
		env = gym.make(self.gym_id)
		self.state_dim = env.observation_space.shape[0]
		self.batch_size = 128
		self.hidden_dim = 128
		self.action_dim = env.action_space.n
		self.discount = 0.99
		self.lr = 7e-4
		self.cuda = True
		self.device = torch.device("cuda" if torch.cuda.is_available() and self.cuda else "cpu")

args = parse_args()
first_half_training_args = parse_args()
first_half_training_args.n_training_steps = first_half_training_args.n_training_steps // 2
second_half_training_args = parse_args()
second_half_training_args.n_training_steps = second_half_training_args.n_training_steps // 2
second_half_training_args.init_steps = 1

/home/plague/miniconda3/envs/torch312/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


## Fast Training Args for Hyperparameter Exploration

For faster hyperparameter exploration, we create a reduced args configuration. 
**Note:** For final evaluation and comparison, use the original `args` configuration.

In [4]:
class parse_args_fast:
	def __init__(self):
		self.n_seeds = 6
		self.n_evaluate_episodes = 5
		self.n_training_steps = 100000
		self.buffer_size = 10000
		self.init_steps = 10000
		self.target_update_freq = 50
		self.eval_freq = 1000
		self.gym_id = "LunarLander-v3"
		env = gym.make(self.gym_id)
		self.state_dim = env.observation_space.shape[0]
		self.batch_size = 512
		self.hidden_dim = 128
		self.action_dim = env.action_space.n
		self.discount = 0.99
		self.lr = 7e-4
		self.cuda = True
		self.device = torch.device("cuda" if torch.cuda.is_available() and self.cuda else "cpu")

args_fast = parse_args_fast()
first_half_training_args_fast = parse_args_fast()
first_half_training_args_fast.n_training_steps = first_half_training_args_fast.n_training_steps // 2
second_half_training_args_fast = parse_args_fast()
second_half_training_args_fast.n_training_steps = second_half_training_args_fast.n_training_steps // 2
second_half_training_args_fast.init_steps = 1

In [5]:
def set_seed(seed):
	torch.manual_seed(seed)
	if torch.cuda.is_available():
		torch.cuda.manual_seed_all(seed)
	np.random.seed(seed)

def weight_init(model):
	torch.nn.init.orthogonal_(model.weight.data)
	model.bias.data.fill_(0.0)

Here we implement the replay buffer.
It has two methods: add one transition to the buffer and sample batch of transitions from the buffer.

In [6]:
class ReplayBuffer:
	def __init__(self, args):
		self.states = np.zeros((args.buffer_size, args.n_seeds, args.state_dim), dtype = np.float32)
		self.actions = np.zeros((args.buffer_size, args.n_seeds), dtype = np.int64)
		self.rewards = np.zeros((args.buffer_size, args.n_seeds), dtype = np.float32)
		self.next_states = np.zeros((args.buffer_size, args.n_seeds, args.state_dim), dtype = np.float32)
		self.terminals = np.zeros((args.buffer_size, args.n_seeds), dtype = np.int64)
		self.idx = 0
		self.current_size = 0
		self.args = args

	def add(self, state, action, reward, next_state, terminal):
		if self.current_size < self.args.buffer_size:
			self.current_size += 1
		self.states[self.idx, :, :] = state
		self.actions[self.idx, :] = action
		self.rewards[self.idx, :] = reward
		self.next_states[self.idx, :, :] = next_state
		self.terminals[self.idx, :] = terminal
		self.idx = (self.idx + 1) % self.args.buffer_size

	def sample(self):
		sample_idxs = np.random.permutation(self.current_size)[:self.args.batch_size]
		states = torch.from_numpy(self.states[sample_idxs]).to(self.args.device)
		actions = torch.from_numpy(self.actions[sample_idxs]).to(self.args.device)
		rewards = torch.from_numpy(self.rewards[sample_idxs]).to(self.args.device)
		next_states = torch.from_numpy(self.next_states[sample_idxs]).to(self.args.device)
		terminals = torch.from_numpy(self.terminals[sample_idxs]).to(self.args.device)

		return states, actions, rewards, next_states, terminals

Here we implement a simple Q network architecture with three layers and ReLU activations.

In [7]:
class QNetwork(torch.nn.Module):
	def __init__(self, args):
		super(QNetwork, self).__init__()
		self.layer_1 = torch.nn.Linear(args.state_dim, args.hidden_dim)
		self.layer_2 = torch.nn.Linear(args.hidden_dim, args.hidden_dim)
		self.layer_3 = torch.nn.Linear(args.hidden_dim, args.action_dim)
		self.relu = torch.nn.ReLU()

		self.layer_1.apply(weight_init)
		self.layer_2.apply(weight_init)
		self.layer_3.apply(weight_init)

	def forward(self, x):
		x = self.relu(self.layer_1(x))
		x = self.relu(self.layer_2(x))
		x = self.layer_3(x)

		return x

Here we provide code for DQN with random exploration.

In [8]:
TRAIN_SEED = 0
EVAL_SEED = 1

class DQN:
	def __init__(self, args):
		self.args = args
		self.discount = self.args.discount
		self.reset()
		self.annealing = False

	# Copying parameters of other DQN class by reference (for half epsion-greedy, half boltzmann task)
	def copy_reference(self, other):
		self.buffer = other.buffer
		self.q_net = other.q_net
		self.q_target = other.q_target
		self.optimizer = other.optimizer

	# Annealing of epsilon and/or temperature
	def anneal(self, step):
		pass

	# Greedy action
	def get_greedy_action(self, states):
		with torch.no_grad():
			action = torch.argmax(self.q_net(states), dim = -1).detach().cpu().numpy()
			return action

	# Exploration action choice
	def explore(self, states):
		# Random action choice
		action = np.random.randint(self.args.action_dim, size = self.args.n_seeds)
		return action

	# Update of the main critic
	def update(self):
		states, actions, rewards, next_states, terminals = self.buffer.sample()
		with torch.no_grad():
			q_next_states = torch.max(self.q_target(next_states), dim = -1)[0]
		ones_tensor = torch.ones_like(terminals).to(self.args.device)
		targets = rewards + (ones_tensor - terminals) * self.discount * q_next_states

		self.optimizer.zero_grad()
		q_values = self.q_net(states).gather(-1, actions.unsqueeze(-1)).squeeze(-1)
		loss = torch.mean((q_values - targets) ** 2)
		loss.backward()
		self.optimizer.step()

	# Update of the targer critic
	def update_target(self):
		self.q_target.load_state_dict(self.q_net.state_dict())

	# Evaluation of the performance on test environments.
	def evaluate(self):
		eval_results = np.zeros(self.args.n_seeds)
		with torch.no_grad():
			eval_env = gym.make_vec(self.args.gym_id, num_envs = self.args.n_seeds, vectorization_mode="sync")
			eval_env.reset(seed = EVAL_SEED)
			for _ in range(self.args.n_evaluate_episodes):
				state, info = eval_env.reset()
				episode_reward = np.zeros(self.args.n_seeds)
				mask = np.ones(self.args.n_seeds)
				while np.sum(mask) > 0:
					action = self.get_greedy_action(torch.tensor(state).to(self.args.device))
					next_state, reward, terminal, truncated, _ = eval_env.step(action)
					episode_reward += mask * reward
					state = next_state
					mask *= (np.ones(self.args.n_seeds) - terminal) * (np.ones(self.args.n_seeds) - truncated)
				eval_results += episode_reward / self.args.n_evaluate_episodes
		return np.mean(eval_results), np.std(eval_results)


	# Resetting the algorithm
	def reset(self):
		self.buffer = ReplayBuffer(self.args)
		self.q_net = QNetwork(self.args).to(self.args.device) # main critic
		self.optimizer = torch.optim.Adam(self.q_net.parameters(), lr = self.args.lr, eps = 1e-5)
		self.q_target = QNetwork(self.args).to(self.args.device) # target critic
		self.update_target()

	# Training loop
	def train(self):
		eval_results_means = np.array([])
		eval_results_stds = np.array([])
		train_env = gym.make_vec(self.args.gym_id, num_envs = self.args.n_seeds, vectorization_mode="sync")
		state, info = train_env.reset(seed = TRAIN_SEED)
		mask = np.ones(self.args.n_seeds)
		for step in range(self.args.n_training_steps):
			action = self.explore(torch.tensor(state).unsqueeze(0).to(self.args.device))
			if self.annealing:
				self.anneal(step)
			next_state, reward, terminal, truncated, _ = train_env.step(action)
			self.buffer.add(state, action, reward, next_state, terminal)
			state = next_state
			if step % self.args.eval_freq == 0:
					print(f"Training step: {step}")
					eval_mean, eval_std = self.evaluate()
					print(f"Eval mean: {eval_mean}; eval_std: {eval_std}")
					eval_results_means = np.append(eval_results_means, eval_mean)
					eval_results_stds = np.append(eval_results_stds, eval_std)
			if step >= self.args.init_steps:
				self.update()
				if step % self.args.target_update_freq == 0:
					self.update_target()
			mask *= (np.ones(self.args.n_seeds) - terminal) * (np.ones(self.args.n_seeds) - truncated)
			if np.sum(mask) == 0:
				state, info = train_env.reset()
				mask = np.ones(self.args.n_seeds)

		return eval_results_means, eval_results_stds


Here we implement functions for plotting.

In [9]:
def smooth(data, weigth = 0.9):
	smooth_data = np.copy(data)
	for index in range(1, len(data)):
		smooth_data[index] = smooth_data[index - 1] * weigth + data[index] * (1.0 - weigth)

	return smooth_data

def plot_smooth(args, result_means, result_stds):
	smooth_result_means = smooth(result_means)
	smooth_result_stds = smooth(result_stds)
	print(smooth_result_means)
	print(smooth_result_stds)
	xs = np.arange(len(result_means)) * args.eval_freq
	print(xs)
	plt.plot(xs, smooth_result_means, color = "blue")
	plt.fill_between(xs, smooth_result_means - smooth_result_stds, smooth_result_means + smooth_result_stds, alpha = 0.2, label = "smoothed_rewards")
	plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
	plt.show()
	plt.clf()

def plot_smooth_many(args, result_means_list, result_stds_list, names_list, colours_list):
	plt.figure(figsize=(12.8, 9.6))
	for result_means, result_stds, name, colour in zip(result_means_list, result_stds_list, names_list, colours_list):
		smooth_result_means = smooth(result_means)
		smooth_result_stds = smooth(result_stds)
		print(smooth_result_means)
		print(smooth_result_stds)
		xs = np.arange(len(result_means)) * args.eval_freq
		print(xs)
		plt.plot(xs, smooth_result_means, color = colour)
		plt.fill_between(xs, smooth_result_means - smooth_result_stds, smooth_result_means + smooth_result_stds, alpha = 0.2, color = colour, label = f"smoothed_rewards_{name}")
		plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
	plt.show()
	plt.clf()

def plot_results(result_mean, result_std):
	plot_smooth(args, result_mean, result_std)

def plot_results_many(result_means_list, result_stds_list, name_list, colours_list):
	plot_smooth_many(args, result_means_list, result_stds_list, name_list, colours_list)

Here we provide code for training across different random seeds.

In [10]:
def train_dqn(dqn):
	set_seed(TRAIN_SEED)
	dqn.reset()
	result_mean, result_std = dqn.train()
	print(result_mean)
	return result_mean, result_std

Here the goal is to implement the epsilon-gredy strategy. With probability epsilon we choose uniformly a random action and with probability 1-epsilon we take the action with the highest Q-value according to the main critic.

In [11]:
class EpsilonGreedyDQN(DQN):
	def __init__(self, args, epsilon):
		super(EpsilonGreedyDQN, self).__init__(args)
		self.epsilon = epsilon # investigate sensitivity

	def explore(self, states):
		action = None
		
		random_values = np.random.random(self.args.n_seeds)
		random_actions = np.random.randint(self.args.action_dim, size = self.args.n_seeds)
		greedy_actions = self.get_greedy_action(torch.squeeze(states))

		action = np.where(random_values < self.epsilon, random_actions, greedy_actions)		
	
		return action

In [12]:
COLOURS = ["red", "green", "blue",
		   "yellow", "magenta",
		   "cyan", "black",
		   "orange"]

# Hyperparameter sensitivity testing for epsilon
means = []
stds = []
epsilons = [0.01, 0.1, 0.3, 0.5]
for epsilon in epsilons:
    model = EpsilonGreedyDQN(args_fast, epsilon)
    m, std = train_dqn(model)
    means.append(m)
    stds.append(std)
names_list = [f"epsilon={e}" for e in epsilons]    
colours_list = COLOURS[:len(epsilons)]
plot_results_many(means, stds, names_list, colours_list)

Training step: 0
Eval mean: -639.6903477514232; eval_std: 59.38467170955477
Training step: 1000
Eval mean: -639.6903477514232; eval_std: 59.38467170955477
Training step: 2000
Eval mean: -639.6903477514232; eval_std: 59.38467170955477
Training step: 3000
Eval mean: -639.6903477514232; eval_std: 59.38467170955477
Training step: 4000
Eval mean: -639.6903477514232; eval_std: 59.38467170955477
Training step: 5000
Eval mean: -639.6903477514232; eval_std: 59.38467170955477
Training step: 6000
Eval mean: -639.6903477514232; eval_std: 59.38467170955477
Training step: 7000
Eval mean: -639.6903477514232; eval_std: 59.38467170955477
Training step: 8000
Eval mean: -639.6903477514232; eval_std: 59.38467170955477
Training step: 9000
Eval mean: -639.6903477514232; eval_std: 59.38467170955477
Training step: 10000
Eval mean: -639.6903477514232; eval_std: 59.38467170955477
Training step: 11000
Eval mean: -100.29406168266894; eval_std: 71.83721305732833
Training step: 12000
Eval mean: -43.196176937931455;

KeyboardInterrupt: 

Here we add to the epsilon-greedy strategy epsilon annealing. We change linearly epsilon from 1.0 to the value final_epsilon during first anneal_steps steps and then it remains on the final_epsilon level.
Such an approach aims to increase the exploration level at the beginning of the training, when the Q-value estimate is poor and thus choosing greedily according to Q is not improving the performance.

### Hyperparameter Sensitivity for Epsilon-Greedy with Annealing

Testing different values of final_epsilon and anneal_steps.

In [ ]:
class EpsilonGreedyWithAnnealingDQN_Tunable(EpsilonGreedyDQN):
	def __init__(self, args, final_epsilon, anneal_steps):
		self.start_epsilon = 1.0
		super(EpsilonGreedyWithAnnealingDQN_Tunable, self).__init__(args, self.start_epsilon)
		self.epsilon = self.start_epsilon
		self.final_epsilon = final_epsilon
		self.annealing = True
		self.anneal_steps = anneal_steps

	def anneal(self, step):
		dif = self.start_epsilon - self.final_epsilon
		self.epsilon = self.start_epsilon - dif*step/self.anneal_steps if step < self.anneal_steps else self.final_epsilon

	def reset(self):
		super(EpsilonGreedyWithAnnealingDQN_Tunable, self).reset()
		self.epsilon = self.start_epsilon

# Test different final_epsilon values
means_final_eps = []
stds_final_eps = []
final_epsilons = [0.01, 0.05, 0.1, 0.2]
for final_eps in final_epsilons:
    model = EpsilonGreedyWithAnnealingDQN_Tunable(args_fast, final_eps, 15000)
    m, std = train_dqn(model)
    means_final_eps.append(m)
    stds_final_eps.append(std)
names_list = [f"final_eps={e}" for e in final_epsilons]    
colours_list = COLOURS[:len(final_epsilons)]
plot_results_many(means_final_eps, stds_final_eps, names_list, colours_list)

In [ ]:
# Test different anneal_steps values
means_anneal_steps = []
stds_anneal_steps = []
anneal_steps_values = [5000, 10000, 15000, 20000]
for anneal_steps in anneal_steps_values:
    model = EpsilonGreedyWithAnnealingDQN_Tunable(args_fast, 0.1, anneal_steps)
    m, std = train_dqn(model)
    means_anneal_steps.append(m)
    stds_anneal_steps.append(std)
names_list = [f"anneal_steps={s}" for s in anneal_steps_values]    
colours_list = COLOURS[:len(anneal_steps_values)]
plot_results_many(means_anneal_steps, stds_anneal_steps, names_list, colours_list)

Alternative approach to the epsilon-greedy strategy is to use so-called boltzmann exploration strategy.
The idea behind this approach is to perform softmax on the Q-values coming from the main critic and then sample from the obtained distribution.
In this approach we use softmax with a temperature, i.e. before applying softmax, we scale all the Q-values by the temperature coefficient (in the literature we usually divide by the temperature, but this is equivallent to scaling by the inverse of the temperature). Large scaling values make the distribution close to the greedy choice, while low scaling values make the distribution close to the uniform one.

### Hyperparameter Sensitivity for Boltzmann

Testing different temperature values.

In [ ]:
class BoltzmannDQN_Tunable(DQN):
	def __init__(self, args, temperature):
		super(BoltzmannDQN_Tunable, self).__init__(args)
		self.temperature = temperature

	def explore(self, states):
		action = None
		with torch.no_grad():
			q_values = self.q_net(states).detach()
		q_values = self.temperature * q_values
		q_probs = torch.softmax(q_values, dim=-1)
		q_probs = q_probs.squeeze()
		action = torch.multinomial(q_probs, num_samples=1).cpu().squeeze(-1).numpy()

		return action

# Test different temperature values
means_temp = []
stds_temp = []
temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]
for temp in temperatures:
    model = BoltzmannDQN_Tunable(args_fast, temp)
    m, std = train_dqn(model)
    means_temp.append(m)
    stds_temp.append(std)
names_list = [f"temp={t}" for t in temperatures]    
colours_list = COLOURS[:len(temperatures)]
plot_results_many(means_temp, stds_temp, names_list, colours_list)

One of the compromises between epsilon-greedy and boltzmann exploration strategy is so-calles max-boltzmann strategy. In this strategy with probability 1-epsilon we choose action greedily, but with probability epsilon we perform the boltzmann choice instead of the uniform random choice.

### Hyperparameter Sensitivity for Max-Boltzmann

Testing different temperature values (epsilon is inherited from parent class).

In [ ]:
class MaxBoltzmannDQN_Tunable(EpsilonGreedyWithAnnealingDQN_Tunable):
	def __init__(self, args, temperature, final_epsilon, anneal_steps):
		super(MaxBoltzmannDQN_Tunable, self).__init__(args, final_epsilon, anneal_steps)
		self.temperature = temperature

	def explore(self, states):
		action = None
		with torch.no_grad():
			q_values = self.q_net(states).detach()

		action_greedy = torch.argmax(q_values, dim = -1).detach().squeeze().cpu().numpy()

		q_values = self.temperature * q_values
		q_probs = torch.softmax(q_values, dim=-1)
		q_probs = q_probs.squeeze()
		action_boltzman = torch.multinomial(q_probs, num_samples=1).cpu().squeeze(-1).numpy()

		random_values = np.random.random(self.args.n_seeds)
		
		action = np.where(random_values < self.epsilon, action_boltzman, action_greedy)		

		return action

# Test different temperature values for Max-Boltzmann
means_maxboltz_temp = []
stds_maxboltz_temp = []
temperatures = [0.05, 0.1, 0.3, 0.5, 1.0]
for temp in temperatures:
    model = MaxBoltzmannDQN_Tunable(args_fast, temp, 0.1, 15000)
    m, std = train_dqn(model)
    means_maxboltz_temp.append(m)
    stds_maxboltz_temp.append(std)
names_list = [f"temp={t}" for t in temperatures]    
colours_list = COLOURS[:len(temperatures)]
plot_results_many(means_maxboltz_temp, stds_maxboltz_temp, names_list, colours_list)

Similarly to adjusting the value of epsilon in epsilon-greedy strategy, we can adjust the temperature in the max-boltzmann and boltzmann strategies: we start we the value start_temperature and linearly increase the value to the final_temperature during temperature_anneal_steps, then the temperature is on the constant level.


### Hyperparameter Sensitivity for Max-Boltzmann with Temperature Annealing

Testing different start_temperature and final_temperature values.

In [ ]:
class MaxBoltzmannWithTemperatureAnnealingDQN_Tunable(MaxBoltzmannDQN_Tunable):
	def __init__(self, args, start_temperature, final_temperature, temperature_anneal_steps, final_epsilon, anneal_steps):
		self.start_temparature = start_temperature
		super(MaxBoltzmannWithTemperatureAnnealingDQN_Tunable, self).__init__(args, self.start_temparature, final_epsilon, anneal_steps)
		self.temperature = self.start_temparature
		self.final_temperature = final_temperature
		self.temperature_anneal_steps = temperature_anneal_steps
		self.annealing = True

	def anneal(self, step):
		super(MaxBoltzmannWithTemperatureAnnealingDQN_Tunable, self).anneal(step)
		dif = self.final_temperature - self.start_temparature
		self.temperature = self.start_temparature + dif*step/self.temperature_anneal_steps if step < self.temperature_anneal_steps else self.final_temperature

	def reset(self):
		super(MaxBoltzmannWithTemperatureAnnealingDQN_Tunable, self).reset()
		self.temperature = self.start_temparature

# Test different temperature ranges
means_maxboltz_temp_anneal = []
stds_maxboltz_temp_anneal = []
temp_configs = [(0.01, 0.1), (0.025, 0.3), (0.05, 0.5), (0.1, 1.0)]
for start_temp, final_temp in temp_configs:
    model = MaxBoltzmannWithTemperatureAnnealingDQN_Tunable(args_fast, start_temp, final_temp, 15000, 0.1, 15000)
    m, std = train_dqn(model)
    means_maxboltz_temp_anneal.append(m)
    stds_maxboltz_temp_anneal.append(std)
names_list = [f"T:{s:.2f}→{f:.1f}" for s, f in temp_configs]    
colours_list = COLOURS[:len(temp_configs)]
plot_results_many(means_maxboltz_temp_anneal, stds_maxboltz_temp_anneal, names_list, colours_list)

### Hyperparameter Sensitivity for Boltzmann with Temperature Annealing

Testing different start_temperature and final_temperature values.

In [ ]:
class BoltzmannWithTemperatureAnnealingDQN_Tunable(BoltzmannDQN_Tunable):
	def __init__(self, args, start_temperature, final_temperature, temperature_anneal_steps):
		self.start_temparature = start_temperature
		super(BoltzmannWithTemperatureAnnealingDQN_Tunable, self).__init__(args, self.start_temparature)
		self.temperature = self.start_temparature
		self.final_temperature = final_temperature
		self.temperature_anneal_steps = temperature_anneal_steps
		self.annealing = True

	def anneal(self, step):
		dif = self.final_temperature - self.start_temparature
		self.temperature = self.start_temparature + dif*step/self.temperature_anneal_steps if step < self.temperature_anneal_steps else self.final_temperature

	def reset(self):
		super(BoltzmannWithTemperatureAnnealingDQN_Tunable, self).reset()
		self.temperature = self.start_temparature

# Test different temperature ranges for Boltzmann with annealing
means_boltz_temp_anneal = []
stds_boltz_temp_anneal = []
temp_configs = [(0.1, 1.0), (0.25, 3.0), (0.5, 5.0), (1.0, 10.0)]
for start_temp, final_temp in temp_configs:
    model = BoltzmannWithTemperatureAnnealingDQN_Tunable(args_fast, start_temp, final_temp, 15000)
    m, std = train_dqn(model)
    means_boltz_temp_anneal.append(m)
    stds_boltz_temp_anneal.append(std)
names_list = [f"T:{s:.1f}→{f:.1f}" for s, f in temp_configs]    
colours_list = COLOURS[:len(temp_configs)]
plot_results_many(means_boltz_temp_anneal, stds_boltz_temp_anneal, names_list, colours_list)

The last exploration idea we want to implement is a combintation of the epsilon-greedy strategy (with epsilon annealing) and the boltzmann strategy.
We could think that at the beginning of the training the boltzmann strategy struggles because the Q-function (the main critic) is not yet well-trained. However, the more critic is trained, the more sense it makes to start using the boltzmann strategy. We would like to verif y this hypoothesis by using in the first half of the training epsilon-greedy strategy (with epsilon annealing) and in the second half of the training switch the exploration strategy to the boltzmann one.

### Hyperparameter Sensitivity for Half Epsilon-Greedy, Half Boltzmann

Testing different final_epsilon (for first half) and temperature (for second half) values.

In [ ]:
# Test different final_epsilon values for the first half
means_half_eps = []
stds_half_eps = []
final_epsilons = [0.05, 0.1, 0.2]
for final_eps in final_epsilons:
    model_1 = EpsilonGreedyWithAnnealingDQN_Tunable(first_half_training_args_fast, final_eps, 7500)
    model_2 = BoltzmannDQN_Tunable(second_half_training_args_fast, 1.0)
    m, std = train_two_halfs_dqn(model_1, model_2)
    means_half_eps.append(m)
    stds_half_eps.append(std)
names_list = [f"final_eps={e}" for e in final_epsilons]    
colours_list = COLOURS[:len(final_epsilons)]
plot_results_many(means_half_eps, stds_half_eps, names_list, colours_list)

In [ ]:
# Test different temperature values for the second half
means_half_temp = []
stds_half_temp = []
temperatures = [0.5, 1.0, 2.0]
for temp in temperatures:
    model_1 = EpsilonGreedyWithAnnealingDQN_Tunable(first_half_training_args_fast, 0.1, 7500)
    model_2 = BoltzmannDQN_Tunable(second_half_training_args_fast, temp)
    m, std = train_two_halfs_dqn(model_1, model_2)
    means_half_temp.append(m)
    stds_half_temp.append(std)
names_list = [f"temp={t}" for t in temperatures]    
colours_list = COLOURS[:len(temperatures)]
plot_results_many(means_half_temp, stds_half_temp, names_list, colours_list)

Here we plot the results of all exploration methods on one plot. However, for drawing conclusions, it might be reasonable to plot some subsets of methods together, for example to compare variants with and without annealing, max-boltzmann with boltzmann, epsilon-greedy, boltzmann and half-epsilon-greedy, half-boltzmann.

## Summary of Hyperparameter Exploration

This notebook now includes hyperparameter sensitivity testing for all exploration strategies:

### Strategies and Hyperparameters Tested:

1. **Epsilon-Greedy**: `epsilon` ∈ {0.01, 0.1, 0.3, 0.5}
2. **Epsilon-Greedy with Annealing**: 
   - `final_epsilon` ∈ {0.01, 0.05, 0.1, 0.2}
   - `anneal_steps` ∈ {5000, 10000, 15000, 20000}
3. **Boltzmann**: `temperature` ∈ {0.1, 0.5, 1.0, 2.0, 5.0}
4. **Boltzmann with Temperature Annealing**: 
   - Temperature ranges: (0.1→1.0), (0.25→3.0), (0.5→5.0), (1.0→10.0)
5. **Max-Boltzmann**: `temperature` ∈ {0.05, 0.1, 0.3, 0.5, 1.0}
6. **Max-Boltzmann with Temperature Annealing**: 
   - Temperature ranges: (0.01→0.1), (0.025→0.3), (0.05→0.5), (0.1→1.0)
7. **Half Epsilon-Greedy, Half Boltzmann**:
   - First half `final_epsilon` ∈ {0.05, 0.1, 0.2}
   - Second half `temperature` ∈ {0.5, 1.0, 2.0}

### Training Configuration:

**Fast Args** (for hyperparameter exploration):
- n_training_steps: 30,000 (vs 100,000)
- n_seeds: 3 (vs 6)
- n_evaluate_episodes: 3 (vs 5)
- eval_freq: 2,000 (vs 1,000)
- buffer_size: 5,000 (vs 10,000)
- init_steps: 5,000 (vs 10,000)

**Original Args** (for final evaluation): Use `args` instead of `args_fast` for final comparisons.

### Next Steps:
1. Run the hyperparameter exploration cells with `args_fast`
2. Analyze the results and select the best hyperparameters
3. Re-run selected configurations with original `args` for final evaluation
4. Compare all methods and draw conclusions